# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

In [3]:
import pandas as pd

# Load the dataset from the URL
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Take a quick look at the columns to understand the data structure
print(df.columns)


Index(['Unnamed: 0', 'Customer', 'State', 'Customer Lifetime Value',
       'Response', 'Coverage', 'Education', 'Effective To Date',
       'EmploymentStatus', 'Gender', 'Income', 'Location Code',
       'Marital Status', 'Monthly Premium Auto', 'Months Since Last Claim',
       'Months Since Policy Inception', 'Number of Open Complaints',
       'Number of Policies', 'Policy Type', 'Policy', 'Renew Offer Type',
       'Sales Channel', 'Total Claim Amount', 'Vehicle Class', 'Vehicle Size',
       'Vehicle Type'],
      dtype='object')


In [4]:
# Standardizing column names for easier access (lowercase, replace spaces with underscores)
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Filter the dataset to include only customers with:
# total_claim_amount below 1000
# response to the last marketing campaign is "Yes"
filtered_df = df[(df['total_claim_amount'] < 1000) & (df['response'] == 'Yes')]

# Check the shape to see how many customers meet this criteria
print(f"Number of customers with total_claim_amount < 1000 and response 'Yes': {filtered_df.shape[0]}")

# Preview the filtered DataFrame
print(filtered_df.head())

Number of customers with total_claim_amount < 1000 and response 'Yes': 1399
    unnamed:_0 customer       state  customer_lifetime_value response  \
3            3  XL78013      Oregon             22332.439460      Yes   
8            8  FM55990  California              5989.773931      Yes   
15          15  CW49887  California              4626.801093      Yes   
19          19  NJ54277  California              3746.751625      Yes   
27          27  MQ68407      Oregon              4376.363592      Yes   

    coverage education effective_to_date employmentstatus gender  ...  \
3   Extended   College           1/11/11         Employed      M  ...   
8    Premium   College           1/19/11         Employed      M  ...   
15     Basic    Master           1/16/11         Employed      F  ...   
19  Extended   College           2/26/11         Employed      F  ...   
27   Premium  Bachelor           2/28/11         Employed      F  ...   

    number_of_open_complaints number_of_polici

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

In [14]:
# Filter customers who responded "Yes"
responded_yes = df[df['response'] == 'Yes']

# Pivot table: average monthly premium and customer lifetime value by policy_type and gender
pivot_avg = responded_yes.pivot_table(
    index=['policy_type', 'gender'],
    values=['monthly_premium_auto', 'customer_lifetime_value'],
    aggfunc='mean'
).round(2)

print("Average Monthly Premium and Customer Lifetime Value by Policy Type and Gender:\n")
print(pivot_avg)

# Pivot table: average total claim amount by policy_type and gender
pivot_claims = responded_yes.pivot_table(
    index=['policy_type', 'gender'],
    values='total_claim_amount',
    aggfunc='mean'
).round(2)

print("\nAverage Total Claim Amount by Policy Type and Gender:\n")
print(pivot_claims)


Average Monthly Premium and Customer Lifetime Value by Policy Type and Gender:

                       customer_lifetime_value  monthly_premium_auto
policy_type    gender                                               
Corporate Auto F                       7712.63                 94.30
               M                       7944.47                 92.19
Personal Auto  F                       8339.79                 99.00
               M                       7448.38                 91.09
Special Auto   F                       7691.58                 92.31
               M                       8247.09                 86.34

Average Total Claim Amount by Policy Type and Gender:

                       total_claim_amount
policy_type    gender                    
Corporate Auto F                   433.74
               M                   408.58
Personal Auto  F                   452.97
               M                   457.01
Special Auto   F                   453.28
               M  

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

In [6]:
# Count customers per state
customers_per_state = df['state'].value_counts()

# Filter states with more than 500 customers
states_over_500 = customers_per_state[customers_per_state > 500]

print("Number of customers per state (only states with > 500 customers):")
print(states_over_500)

Number of customers per state (only states with > 500 customers):
state
California    3552
Oregon        2909
Arizona       1937
Nevada         993
Washington     888
Name: count, dtype: int64


4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

In [7]:
clv_stats = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(['max', 'min', 'median'])
print(clv_stats)


                                     max          min       median
education            gender                                       
Bachelor             F       73225.95652  1904.000852  5640.505303
                     M       67907.27050  1898.007675  5548.031892
College              F       61850.18803  1898.683686  5623.611187
                     M       61134.68307  1918.119700  6005.847375
Doctor               F       44856.11397  2395.570000  5332.462694
                     M       32677.34284  2267.604038  5577.669457
High School or Below F       55277.44589  2144.921535  6039.553187
                     M       83325.38119  1940.981221  6286.731006
Master               F       51016.06704  2417.777032  5729.855012
                     M       50568.25912  2272.307310  5579.099207


In [12]:
# Making numbers more readable
clv_stats_rounded = clv_stats.round(2)

# Format with thousands separators
clv_stats_formatted = clv_stats_rounded.applymap(lambda x: f"{x:,.2f}")

print(clv_stats_formatted)


                                   max       min    median
education            gender                               
Bachelor             F       73,225.96  1,904.00  5,640.51
                     M       67,907.27  1,898.01  5,548.03
College              F       61,850.19  1,898.68  5,623.61
                     M       61,134.68  1,918.12  6,005.85
Doctor               F       44,856.11  2,395.57  5,332.46
                     M       32,677.34  2,267.60  5,577.67
High School or Below F       55,277.45  2,144.92  6,039.55
                     M       83,325.38  1,940.98  6,286.73
Master               F       51,016.07  2,417.78  5,729.86
                     M       50,568.26  2,272.31  5,579.10


C:\Users\jdhernandezd\AppData\Local\Temp\ipykernel_5112\4277738231.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  clv_stats_formatted = clv_stats_rounded.applymap(lambda x: f"{x:,.2f}")


### Conclusion

1) The highest maximum customer lifetime value (CLV) is found among males with "High School or Below" education, indicating that higher education does not necessarily correlate with higher maximum CLV in this dataset.

2) Median CLV values are relatively consistent across education levels, with the "High School or Below" group showing some of the highest median CLVs, suggesting stable customer value in this segment.

3) Customers with a Doctorate degree have the lowest maximum CLV, which might be influenced by smaller sample size or different customer behaviors.

4) Gender differences are present but not pronounced; both males and females exhibit similar median CLVs across education levels.

In general, education level by itself does not strongly determine customer lifetime value in this dataset. Notably, some groups with lower formal education exhibit high and steady CLVs, emphasizing the need to evaluate additional variables when analyzing customer value.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
# your code goes here